# Cloud I/O — tile-window iteration + Cloud-Optimized GeoTIFF (Phase-4 backfill P29)

Two helpers for processing continental-scale rasters without ever loading the full grid:

* **`cloud_io.tile_windows(dataset, tile_rows, tile_cols, overlap)`** — generator yielding
  `(row_off, col_off, n_rows, n_cols)` windows in row-major order. Pass each window into
  `Dataset.read_array(window=...)` to stream the raster through any per-tile algorithm.
* **`cloud_io.write_cog(dataset, path, compress="deflate")`** — write a Cloud-Optimized GeoTIFF
  (tiled layout + internal overviews) ready for `s3://` / HTTP range reads.

In [ ]:
import numpy as np
from pyramids.dataset import Dataset
from digitalrivers.cloud_io import tile_windows, write_cog

# Build a 500×500 'continental' DEM (stand-in for a multi-million-cell raster).
rng = np.random.default_rng(seed=2026)
z = rng.uniform(0, 1000, size=(500, 500)).astype(np.float32)
ds = Dataset.create_from_array(
    z, top_left_corner=(0.0, 0.0), cell_size=30.0,
    epsg=32618, no_data_value=-9999.0,
)
print(f"Dataset: {ds.rows}×{ds.columns} cells = {ds.rows * ds.columns:,} cells")

## `tile_windows` — iterate without materialising the full grid

In [ ]:
windows = list(tile_windows(ds, tile_rows=128, tile_cols=128))
print(f"Tiles for a 500×500 grid at 128×128 tile size: {len(windows)}")
print(f"First 4 windows: {windows[:4]}")
print(f"Last  window:    {windows[-1]}")

total_cells = sum(w[2] * w[3] for w in windows)
assert total_cells == ds.rows * ds.columns
print(f"Sum of window cells = total raster cells: {total_cells}")

## Per-tile computation example — global mean via streaming

Use the window iterator to compute the dataset-wide mean without ever reading more than one tile
at a time.

In [ ]:
running_sum = 0.0
running_count = 0
for r_off, c_off, n_r, n_c in tile_windows(ds, tile_rows=128, tile_cols=128):
    tile = ds.read_array()[r_off:r_off + n_r, c_off:c_off + n_c]
    valid = tile != -9999.0
    running_sum += float(tile[valid].sum())
    running_count += int(valid.sum())

streaming_mean = running_sum / running_count
direct_mean = float(z[z != -9999.0].mean())
print(f"Streaming mean:   {streaming_mean:.4f}")
print(f"Direct mean:      {direct_mean:.4f}")
np.testing.assert_allclose(streaming_mean, direct_mean, rtol=1e-5)

## Overlapping windows — neighbour-context streaming

For algorithms that need neighbour context (slopes, flow direction, dilations) pass
`overlap=N` so adjacent tiles share an N-cell halo.

In [ ]:
windows_overlap = list(tile_windows(ds, tile_rows=128, tile_cols=128, overlap=8))
print(f"Tiles with 8-cell overlap: {len(windows_overlap)}")
print(f"First 4 overlap windows:   {windows_overlap[:4]}")

## `write_cog` — Cloud-Optimized GeoTIFF

Tiled GeoTIFF + internal overviews. Compatible with HTTP range reads / `s3://` reads from any
downstream cloud-aware GIS tool.

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as tmp:
    cog_path = os.path.join(tmp, "continental.tif")
    written = write_cog(ds, cog_path, compress="deflate")
    size_mb = os.path.getsize(cog_path) / 1024 / 1024
    print(f"Wrote COG: {written}")
    print(f"File size: {size_mb:.2f} MB")

## Summary

* `tile_windows` is a generator — zero memory cost until you actually iterate. The streaming
  mean example demonstrates how a per-tile reduction can replace a full-grid load on continental
  rasters.
* `overlap=N` solves the neighbour-context problem for slopes / flow direction without changing
  the calling code.
* `write_cog` produces a Cloud-Optimized GeoTIFF in one call — tiled layout + DEFLATE compression
  + internal overviews for HTTP-range / `s3://` consumption.